In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import CRPS.CRPS as pscore

import multiprocessing as mp
mp.set_start_method('spawn')


pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

sys.path.append('../../../Evaluation/')
from normal_evaluation.dumas_evaluation import *

In [2]:
with open('./dumas_model.pickle', 'rb') as f:
    dumas_model = pickle.load(f)

with open('./dumas_model_no_resource.pickle', 'rb') as f:
    dumas_model_no_resource = pickle.load(f)

In [3]:
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]
n_processes = 64

N = 1000

In [4]:
#with open('../transformed_event_logs/Helpdesk_train.pickle', 'rb') as f:
#    train_data = pickle.load(f)

with open('../../transformed_event_logs/Helpdesk_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

test_data['Case ID'] = test_data['Case ID'].astype(str)
test_data['case:concept:name'] = test_data['Case ID']
test_data['time:timestamp_start'] = test_data['Complete Timestamp_start']
test_data['time:timestamp_complete'] = test_data['Complete Timestamp_complete']


evaluator = conduct_evaluation.ConductEvaluation(dumas_model, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     train_data, n=N, n_processes = n_processes)
likelihoods_train_A_R = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

np.mean([v.ln() for v in likelihoods_train_A_R[0].values()])

np.mean(get_pscores(likelihoods_train_A_R))

In [5]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                 },
                                     test_data, n=N, n_processes = n_processes)
likelihoods_test_A_R = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

  0%|          | 0/919 [00:00<?, ?it/s]

100%|██████████| 919/919 [00:09<00:00, 92.62it/s] 


In [6]:
np.mean([v.ln() for v in likelihoods_test_A_R[0].values()])

Decimal('-4.004931818307991827099154852')

In [7]:
np.mean(get_pscores(likelihoods_test_A_R))

np.float64(649271.239568482)

evaluator = conduct_evaluation.ConductEvaluation(dumas_model_no_resource, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     train_data, n=N, n_processes = n_processes)
likelihoods_train_A = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

np.mean([v.ln() for v in likelihoods_train_A[0].values()])

np.mean(get_pscores(likelihoods_train_A))

In [8]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model_no_resource, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                 },
                                     test_data, n=N, n_processes = n_processes)
likelihoods_test_A = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

100%|██████████| 919/919 [00:08<00:00, 102.75it/s]


In [9]:
np.mean([v.ln() for v in likelihoods_test_A[0].values()])

Decimal('-4.031398574807205615450216285')

In [10]:
np.mean(get_pscores(likelihoods_test_A))

np.float64(667417.494408731)

In [11]:
results = {
    'dumas_test_A' : likelihoods_test_A,
    'dumas_test_A_R' : likelihoods_test_A_R,
    #'dumas_train_A' : likelihoods_train_A,
    #'dumas_train_A_R' : likelihoods_train_A_R
}

with open('./artificial_dumas_evaluation_.pickle', 'wb') as handle:
    pickle.dump(results, handle)